In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install xgboost lightgbm "mlflow<3"

In [3]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025


In [7]:
import sqlite3
import pandas as pd

# base_folder should be defined in your environment (e.g., base_folder = ".")
conn = sqlite3.connect(f"{base_folder}/data/heart.db")

heart_data = pd.read_sql_query(
    """
    SELECT
        p.patient_id,
        p.age,
        p.sex,
        p.cp_id,
        p.trestbps,
        p.chol,
        p.fbs,
        p.ecg_id,
        p.thalach,
        p.exang,
        p.slope_id,
        p.ca,
        p.thal_id,
        d.target
    FROM patient AS p
    JOIN diagnosis AS d
        ON p.patient_id = d.patient_id
    ORDER BY p.patient_id
    """,
    conn,
)
conn.close()

# Display the first few rows to verify oldpeak is gone and target is joined
heart_data.head()

,patient_id,age,sex,cp_id,trestbps,chol,fbs,ecg_id,thalach,exang,slope_id,ca,thal_id,target
0,0,52,1,1,125,212,0,1,168,0,1,2,1,0
1,1,53,1,1,140,203,1,2,155,1,2,0,1,0
2,2,70,1,1,145,174,0,1,125,1,2,0,1,0
3,3,61,1,1,148,203,0,1,161,0,1,1,1,0
4,4,62,0,1,138,294,1,1,106,0,3,3,2,0


In [5]:
import os
import numpy as np
import pandas as pd
import time

from dotenv import load_dotenv

from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline

import mlflow
from mlflow.models import infer_signature
import joblib

# Import shared components from heart_pipeline.py
from heart_pipeline import (
    build_preprocessing,
    make_estimator_for_name,
)

start_time = time.monotonic()

# =============================================================================
# STEP 1: Build Full ML Preprocessing Pipeline
# =============================================================================
# This now contains ColumnDropper('patient_id') + selection of [ca, cp_id, thal_id]
preprocessing = build_preprocessing()
print("✓ STEP 1: Simplified Heart data preprocessing pipeline created.")


# =============================================================================
# STEP 2: Split Data into Stratified Train and Test Sets
# =============================================================================
# KEEPING patient_id here so the pipeline's ColumnDropper has something to drop,
# and MLflow logging remains stable.
X = heart_data.drop(["target"], axis=1)
y = heart_data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"✓ STEP 2: Stratified split done. Features used: {X_train.columns.tolist()}")


# =============================================================================
# STEP 3: Define 4 Model Pipelines (WITHOUT PCA)
# =============================================================================
model_names = ["logistic", "histgradientboosting", "xgboost", "lightgbm"]
models = {}
for name in model_names:
    est = make_estimator_for_name(name)
    # The 'preprocessing' step handles the dropping of patient_id and selection of 3 features
    models[name] = make_pipeline(preprocessing, est)

print("✓ STEP 3: 4 baseline classification pipelines defined.")


# =============================================================================
# STEP 4: Configure MLflow (e.g., Dagshub) via .env
# =============================================================================
load_dotenv(
    dotenv_path="/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025/notebooks/.env",
    override=True
)

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("heart_disease_classification_v3_top3")

print("✓ STEP 4: MLflow configured.")


# =============================================================================
# STEP 5 & 6: Unified Training Loop
# =============================================================================
results = {}

def evaluate_and_log(name, pipeline, uses_pca=False):
    run_name = f"{name}{'_pca' if uses_pca else '_baseline'}"
    print(f"\nTraining: {run_name}")

    # Cross-validation using F1 score
    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=5, scoring="f1", n_jobs=-1
    )
    cv_f1 = cv_scores.mean()

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    test_f1 = f1_score(y_test, y_pred)
    test_acc = accuracy_score(y_test, y_pred)

    # Store results
    results[run_name] = {"pipeline": pipeline, "test_f1": test_f1, "cv_f1": cv_f1}

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_family", name)
        mlflow.log_param("uses_pca", uses_pca)
        mlflow.log_param("features_selected", "ca, cp_id, thal_id")

        # Log metrics (F1 score will appear as a column in DagsHub)
        mlflow.log_metric("cv_f1", cv_f1)
        mlflow.log_metric("f1_score", test_f1)
        mlflow.log_metric("test_accuracy", test_acc)

        # Log model signature
        signature = infer_signature(X_train, pipeline.predict(X_train))
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model",
            signature=signature,
            registered_model_name=f"heart_{run_name}"
        )

# Run Baselines
for name, pipe in models.items():
    evaluate_and_log(name, pipe, uses_pca=False)

# Run PCA variants
for name in model_names:
    pca_pipe = make_pipeline(preprocessing, PCA(n_components=0.95), make_estimator_for_name(name))
    evaluate_and_log(name, pca_pipe, uses_pca=True)


# =============================================================================
# STEP 7 & 8: Global Best and Save
# =============================================================================
global_best_name = max(results, key=lambda k: results[k]["test_f1"])
global_best_pipeline = results[global_best_name]["pipeline"]

print(f"\nWINNER: {global_best_name} with F1: {results[global_best_name]['test_f1']:.4f}")

os.makedirs(f"{base_folder}/models", exist_ok=True)
joblib.dump(global_best_pipeline, f"{base_folder}/models/global_best_heart_model.pkl")

print(f"Elapsed: {int((time.monotonic() - start_time) // 60)}m {(time.monotonic() - start_time) % 60:.1f}s")

✓ STEP 1: Simplified Heart data preprocessing pipeline created.
✓ STEP 2: Stratified split done. Features used: ['patient_id', 'age', 'sex', 'cp_id', 'trestbps', 'chol', 'fbs', 'ecg_id', 'thalach', 'exang', 'slope_id', 'ca', 'thal_id']
✓ STEP 3: 4 baseline classification pipelines defined.


2026/02/16 06:48:11 INFO mlflow.tracking.fluent: Experiment with name 'heart_disease_classification_v3_top3' does not exist. Creating a new experiment.


✓ STEP 4: MLflow configured.

Training: logistic_baseline


/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'heart_logistic_baseline'.
2026/02/16 06:48:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: heart_logistic_baseline, version 1
Created ver

🏃 View run logistic_baseline at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0/runs/0fcb84ec15454190bdc3097029e13620
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0

Training: histgradientboosting_baseline


...

🏃 View run histgradientboosting_pca at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0/runs/06157735ab0b4b6bbb818e994506e15a
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0

Training: xgboost_pca


/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'heart_xgboost_pca'.
2026/02/16 06:50:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: heart_xgboost_pca, version 1
Created version '1' of 

🏃 View run xgboost_pca at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0/runs/63b353392eb14eab93c639a6e5fed5ef
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0

Training: lightgbm_pca
[LightGBM] [Info] Number of positive: 421, number of negative: 399
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000231 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 169
[LightGBM] [Info] Number of data points in the train set: 820, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.513415 -> initscore=0.053671
[LightGBM] [Info] Start training from score 0.053671


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing

🏃 View run lightgbm_pca at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0/runs/dbef6893743d43a3a49fbfe98ac56c8c
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/0

WINNER: lightgbm_baseline with F1: 0.8638
Elapsed: 2m 38.8s
